<a href="https://colab.research.google.com/github/Nijimbere722/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4



### Name: Monia Nijimbere
### Student ID: 67912028




## Part 1.1

In [3]:
import os
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")
from openai import OpenAI
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"
print("Client ready.")

Client ready.


In [4]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response


# Test call
resp = ask_llm("What are three things a microfinance loan officer should look for in a loan application letter?")
answer_text = resp.choices[0].message.content
print(answer_text)
print("\n Token usage")
print(resp.usage)

A microfinance loan officer should look for the following three things in a loan application letter:

1. **Clear Business Plan and Use of Funds**: The loan application letter should clearly outline the borrower's business plan, including the type of business, target market, and expected income. The letter should also specify how the loan funds will be used, such as for working capital, equipment purchases, or expansion. This helps the loan officer understand the borrower's intentions and assess the viability of the business.

2. **Repayment Capacity and Cash Flow**: The loan application letter should provide information about the borrower's repayment capacity, including their current income, expenses, and cash flow. This helps the loan officer assess the borrower's ability to repay the loan, including the interest and principal, in a timely manner.

3. **Collateral and Credit History**: The loan application letter should provide information about the borrower's credit history, includin

1. System vs. user roles:
The system role sets the model's overall behavior, persona, and rules for the whole conversation. It's set once and stays in effect. The user role is the actual input or question for that specific turn.

* Example for system: "You are an assistant to a microfinance loan officer. Be factual and neutral, and never invent details that ar not in the letter."
Example for user: "Summarize this loan application:"letter text"

2. What is a token:
a small chunk of text such as a word, part of a word, a number, or a punctuation mark.

* API providers bill per token rather than per request because the actual computational cost is driven by how much text the model has to read and generate, not by how many times you called the API. A one-word request and a 500-word request are both "one request," but the second one takes far more compute, so billing by token ties the cost directly to the actual work done.

## Part 1.2

In [5]:
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = [ask_llm(question, temperature=0.0).choices[0].message.content for _ in range(5)]
high_temp_answers = [ask_llm(question, temperature=1.2).choices[0].message.content for _ in range(5)]

print(" Temperature = 0.0 ")
for i, a in enumerate(low_temp_answers, 1):
    print(f"\n[{i}] {a}")

print("\n\n Temperature = 1.2 ")
for i, a in enumerate(high_temp_answers, 1):
    print(f"\n[{i}] {a}")

 Temperature = 0.0 

[1] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving money being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **MarketMate Savings**: This name positions the savings product as a trusted companion for market traders.
5. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for the savings product.
6. **Accra Trader's Fund**: This name is straightforward and clearly communicates the product's target audience.
7. **Sukuu Savings**: "Sukuu" is a Ghanaian word for "save" or "keep", which could make the product more relatable and accessible to market traders.

Choose the one that resonates 

* At temperature 0.0, the five answers were nearly identical across runs, the same names ("Makola Save," "Sika Su," "Kokroko Savings") reappeared almost word-for-word each time, and two runs were exact duplicates. At temperature 1.2, the five answers varied much more different Ghanaian languages drawn on, different name structures, and more unusual suggestions. For the loan decision-support system, low temperature (0–0.2) is the right choice, because summarizing and extracting data from a loan letter needs to be consistent and factual , a loan officer shouldn't get a different summary of the same letter each time it's processed.

## Section 2

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")
print("\n L002\n", LETTERS["L002"])
print("\nL006 \n", LETTERS["L006"])

6 letters loaded.

 L002
 Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.

L006 
 Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.


## Section 3

### Part 3.1

In [7]:
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"

for lid in ["L002", "L006"]:
    out = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]))
    print(f" {lid} (V1)\n{out.choices[0].message.content}\n")

 L002 (V1)
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He promises to repay the loan as soon as his business picks up, likely after the festive season, despite not having collateral at the moment.

 L006 (V1)
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his self-proclaimed trustworthiness.



In [8]:
SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan application letters factually and neutrally in 3-4 sentences. "
    "Only include information explicitly stated in the letter. "
    "Do not invent, infer, or embellish any detail not present in the source text. "
    "Do not offer an opinion on whether the loan should be approved."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

def summarize_letter(letter_text):
    resp = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0,
    )
    return resp.choices[0].message.content

for lid in ["L002", "L006"]:
    out = summarize_letter(LETTERS[lid])
    print(f" {lid} (V2) \n{out}\n")

 L002 (V2) 
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He states that the loan is needed to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to offer, but is requesting assistance with the loan.

 L006 (V2) 
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He is 22 years old and claims to be "full of energy" and "business minded" as stated by his friends. Kofi has not yet started any of these businesses. He intends to repay the loan in one year and does not have collateral, but describes himself as trustworthy.



1. Difference between V1 and V2:

* V1 ( "summarize this" prompt) added small opinions that weren't in the letter. For L006, it said Kofi "claims" to be business-minded and is "relying on" his trustworthiness,these words sound doubtful, like the summary doesn't fully believe him, even though the letter never said that.

* V2 (with clear instructions to be factual and neutral) fixed this. It still mentions that Kofi says he is "business minded," but it clearly says this came from his friends ("as stated by his friends") instead of sounding doubtful. It reports what the letter says without adding a hidden opinion.

2. Why "no invented details" matters:

* A loan officer may only read the summary and not the full letter. If the summary adds or changes small details, the officer could make a decision based on wrong information. When an AI adds information that was never actually said, this is called hallucination. It is a serious problem for something like a loan system, because it could affect a real financial decision.

## Part 3.2

In [9]:
EXTRACT_SYSTEM_PROMPT = (
    "You are a data extraction engine for a microfinance loan system. "
    "You extract structured fields from loan application letters. "
    "Return ONLY a valid JSON object — no prose, no markdown code fences, no explanation. "
    "The JSON object must have EXACTLY these keys: "
    "applicant_name (string), amount_ghs (number), purpose (string), "
    "monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), "
    "repayment_months (number or null). "
    "If a field is not stated in the letter, use null. Do not guess or infer a value."
)

# One worked few-shot example
FEW_SHOT_LETTER = (
    "Dear Sir, I am Ama Serwaa, a hairdresser in Cape Coast. I request GHS 5,000 to buy "
    "new dryers. My shop makes about GHS 600 profit monthly. I can repay GHS 300 monthly "
    "for 18 months. I have no guarantor yet."
)
FEW_SHOT_ANSWER = (
    '{"applicant_name": "Ama Serwaa", "amount_ghs": 5000, '
    '"purpose": "buy new dryers", "monthly_profit_ghs": 600, '
    '"has_collateral_or_guarantor": false, "repayment_months": 18}'
)

EXTRACT_PROMPT = """Here is a worked example:

Letter:
{fewshot_letter}

JSON:
{fewshot_answer}

Now extract the same fields from this letter. Return ONLY the JSON object.

Letter:
{letter}

JSON:"""


import json

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEW_SHOT_LETTER,
        fewshot_answer=FEW_SHOT_ANSWER,
        letter=letter_text,
    )
    resp = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=0, max_tokens=300)
    raw = resp.choices[0].message.content.strip()
    # Strip ```json ... ``` fences if the model added them anyway
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.replace("json\n", "", 1) if raw.startswith("json") else raw
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f" Failed to parse JSON. Raw output was:\n{raw}")
        return None

In [17]:
import pandas as pd

rows = []
for lid, letter in LETTERS.items():
    fields = extract_fields(letter)
    if fields is not None:
        fields = {"letter_id": lid, **fields}
    else:
        fields = {"letter_id": lid}
    rows.append(fields)

extraction_df = pd.DataFrame(rows).set_index("letter_id")
extraction_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


In [18]:
NO_NULL_SYSTEM_PROMPT = (
    "You are a data extraction engine for a microfinance loan system. "
    "You extract structured fields from loan application letters. "
    "Return ONLY a valid JSON object with EXACTLY these keys: "
    "applicant_name (string), amount_ghs (number), purpose (string), "
    "monthly_profit_ghs (number), has_collateral_or_guarantor (boolean), "
    "repayment_months (number)."

)

test_prompt = EXTRACT_PROMPT.format(
    fewshot_letter=FEW_SHOT_LETTER,
    fewshot_answer=FEW_SHOT_ANSWER,
    letter=LETTERS["L002"],
)

resp = ask_llm(test_prompt, system_prompt=NO_NULL_SYSTEM_PROMPT, temperature=0, max_tokens=300)
print(resp.choices[0].message.content)

{"applicant_name": "Kwame Boateng", "amount_ghs": 25000, "purpose": "repair trotro engine and settle personal debts", "monthly_profit_ghs": null, "has_collateral_or_guarantor": false, "repayment_months": null}


1. Why the few-shot example can't come from the six letters:

* If the example letter were one of the six we're actually processing, the model could just repeat an answer it already saw, instead of really learning to apply the pattern to new text. Using a made-up letter (Ama Serwaa) makes sure the model is genuinely generalizing the task.

2. What happened without the "use null" instruction:

* I tested this by removing the "use null, do not guess" line and running it on L002, which never states a profit figure or repayment period. In this run, the model still correctly returned null for both missing fields, even without the explicit instruction. This suggests the model has some built-in tendency to avoid guessing when a value isn't stated. but I still keep the explicit instruction in the final prompt, because relying on the model's default behavior isn't reliable enough for a real system; an explicit instruction removes the guesswork and makes the behavior consistent across different letters and situations, not just the one I tested.

3. Why temperature=0 for extraction but not creative tasks:

* Extraction has one correct answer per field, either the letter states it or it doesn't, so we want the same, most literal answer every time, which is what temperature=0 gives. Creative tasks like the product-naming exercise in Part 1.2 have many equally good answers, so some randomness there produces useful variety instead of just repeating the same idea.

### Part 3.3

In [19]:
BRIEF_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer in Ghana. Your job is to prepare "
    "a decision-support brief — NOT a decision. Final approval or rejection is always made "
    "by a human loan officer. Never output the words 'approve' or 'reject'. "
    "Base every point strictly on the letter and the extracted data provided; do not invent facts."
)

BRIEF_PROMPT = """Letter:
{letter}

Extracted data:
{extracted_json}

Write a decision-support brief with exactly these four sections:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review" — NOT approve/reject)
"""

def generate_brief(letter_text, extracted_fields):
    prompt = BRIEF_PROMPT.format(
        letter=letter_text,
        extracted_json=json.dumps(extracted_fields, indent=2) if extracted_fields else "{}",
    )
    resp = ask_llm(prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0.3, max_tokens=500)
    return resp.choices[0].message.content

briefs = {}
for lid, letter in LETTERS.items():
    extracted = extraction_df.loc[lid].dropna().to_dict() if lid in extraction_df.index else {}
    briefs[lid] = generate_brief(letter, extracted)

for lid in ["L001", "L002", "L006"]:
    print(f" {lid} ")
    print(briefs[lid])
    print()

 L001 
## 1. Strengths
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, indicating a consistent income stream.
* Akosua has demonstrated savings discipline through the susu scheme, accumulating GHS 2,500 over two years without missing a contribution.
* She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.
* The applicant has proposed a repayment plan of GHS 450 monthly over 20 months, which is roughly half of her monthly profit, suggesting a manageable repayment burden.

## 2. Risks / red flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit and savings, which might pose a risk if the business expansion does not yield expected returns.
* There is no information provided about the current market demand for frozen foods at Makola Market or the competition in this area, w

1. Comparing a strong application (L001) vs a weak one (L006):

* For L001 (Akosua), the brief correctly picked out real strengths, 12 years running her stall, a steady GHS 900 monthly profit, two years of consistent susu savings, and a guarantor. It also found real risks, like the loan being large compared to her monthly profit. For L006 (Kofi), the brief correctly flagged that he has no experience, no collateral, and an overly optimistic one-year repayment plan based on businesses that don't exist yet. It also noticed that "business-minded" is just his own opinion, not proof. So, the system correctly told a strong application apart from a weak one, and the reasons it gave matched what was actually in the letters.

2. Why we forbid "approve"/"reject":

* Practically, the model only has one letter to go on, it doesn't know the bank's full lending rules, the applicant's credit history, or other information a real officer would check. Ethically, letting an AI make the final call on someone's loan removes human responsibility from a decision that seriously affects people's lives if the system is wrong or biased, no one is accountable. That's why every brief ends with a next step like "invite for interview," not a yes/no answer.

In [20]:
prompts_py_content = f'''"""
Final prompt templates for Lab 4 — LLM Decision Support System.

Evolution notes:
- Summarization: started as a bare "Summarize this:" (V1), which produced summaries
  that subtly editorialized (e.g. sounding doubtful about an applicant's claims).
  V2 adds a role, a length limit (3-4 sentences), and a "no invented details / stay
  neutral" instruction, which fixed this.
- Extraction: uses an explicit JSON schema, one few-shot example (written from scratch,
  not from the working dataset, to avoid leaking answers), and an explicit
  "use null, do not guess" instruction as a safety net for consistent behavior.
- Brief: explicitly instructed to never output "approve"/"reject" and to keep the
  human in the loop, per the lab's decision-support (not decision-making) requirement.
"""

SUMMARY_SYSTEM_PROMPT = {SUMMARY_SYSTEM_PROMPT_V2!r}
SUMMARY_PROMPT = {SUMMARY_PROMPT_V2!r}

EXTRACT_SYSTEM_PROMPT = {EXTRACT_SYSTEM_PROMPT!r}
EXTRACT_PROMPT = {EXTRACT_PROMPT!r}

BRIEF_SYSTEM_PROMPT = {BRIEF_SYSTEM_PROMPT!r}
BRIEF_PROMPT = {BRIEF_PROMPT!r}
'''

with open("prompts.py", "w") as f:
    f.write(prompts_py_content)

print("prompts.py written.")

prompts.py written.


In [21]:
from google.colab import files
files.download("prompts.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Section 4

## Part4.1

In [23]:
import numpy as np

gold_ids = list(GOLD.keys())
fields_to_check = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
                    "has_collateral_or_guarantor", "repayment_months"]

def values_match(field, predicted, gold):
    predicted_missing = predicted is None or (isinstance(predicted, float) and np.isnan(predicted))
    gold_missing = gold is None
    if predicted_missing and gold_missing:
        return True
    if predicted_missing or gold_missing:
        return False
    if field in ("applicant_name", "purpose"):
        return str(predicted).strip().lower() == str(gold).strip().lower()
    return predicted == gold

accuracy_rows = []
for field in fields_to_check:
    row = {"field": field}
    correct = 0
    for lid in gold_ids:
        predicted = extraction_df.loc[lid, field] if lid in extraction_df.index else None
        gold_val = GOLD[lid][field]
        match = values_match(field, predicted, gold_val)
        row[lid] = "MATCH" if match else f"MISMATCH ({predicted!r} vs {gold_val!r})"
        correct += match
    row["accuracy"] = f"{correct}/{len(gold_ids)}"
    accuracy_rows.append(row)

accuracy_df = pd.DataFrame(accuracy_rows).set_index("field")
accuracy_df

,L001,L003,L006,accuracy
field,,,,
applicant_name,MATCH,MATCH,MATCH,3/3
amount_ghs,MATCH,MATCH,MATCH,3/3
purpose,MISMATCH ('buy a deep freezer and expand into ...,MISMATCH ('purchase two industrial sewing mach...,"MISMATCH ('start a car washing business, a pro...",0/3
monthly_profit_ghs,MATCH,MATCH,MATCH,3/3
has_collateral_or_guarantor,MATCH,MATCH,MATCH,3/3
repayment_months,MATCH,MATCH,MATCH,3/3


## Part 4.2

In [24]:
import json as _json

def run_reliability_test(letter_id, temperature, n_runs=5):
    results = []
    valid_json_count = 0
    for _ in range(n_runs):
        if temperature == 0:
            fields = extract_fields(LETTERS[letter_id])
        else:
            prompt = EXTRACT_PROMPT.format(
                fewshot_letter=FEW_SHOT_LETTER,
                fewshot_answer=FEW_SHOT_ANSWER,
                letter=LETTERS[letter_id],
            )
            resp = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT,
                            temperature=temperature, max_tokens=300)
            raw = resp.choices[0].message.content.strip().strip("`")
            try:
                fields = _json.loads(raw)
            except _json.JSONDecodeError:
                fields = None

        if fields is not None:
            valid_json_count += 1
            results.append(_json.dumps(fields, sort_keys=True))
        else:
            results.append(None)

    unique_valid = len(set(r for r in results if r is not None))
    return valid_json_count, unique_valid, results

for temp in [0.0, 1.0]:
    valid_count, unique_count, raw_results = run_reliability_test("L004", temp)
    print(f" Temperature = {temp} ")
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique results among valid runs: {unique_count} (1 = fully consistent)")
    for i, r in enumerate(raw_results, 1):
        print(f"  Run {i}: {r}")
    print()

 Temperature = 0.0 
Valid JSON: 5/5
Unique results among valid runs: 1 (1 = fully consistent)
  Run 1: {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
  Run 2: {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
  Run 3: {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
  Run 4: {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
  Run 5: {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpos

## Part 4.3

In [25]:
# Test 1: ask about a detail NOT present in a letter
test1_question = "Based on this letter, what is the applicant's credit score?"
test1_prompt = f"Letter:\n{LETTERS['L003']}\n\nQuestion: {test1_question}\nIf the answer is not stated in the letter, say so explicitly."
test1_resp = ask_llm(test1_prompt, system_prompt=SUMMARY_SYSTEM_PROMPT_V2, temperature=0)
print(" Test 1: asking about an absent detail")
print(test1_resp.choices[0].message.content)
print()

# Test 2: feed the extractor something irrelevant
IRRELEVANT_TEXT = (
    "Weather report for Accra, 9 August 2026: partly cloudy, high of 31C, "
    "chance of afternoon showers near the coast, humidity around 78%."
)
test2_result = extract_fields(IRRELEVANT_TEXT)
print("Test 2: extracting from an irrelevant (weather) text ")
print(test2_result)

 Test 1: asking about an absent detail
The applicant's credit score is not stated in the letter.

Test 2: extracting from an irrelevant (weather) text 
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


1. Accuracy: My extraction got 5 out of 6 fields perfectly right when checked against the gold answers. Only purpose looked wrong, but that's because it's written in different words that mean the same thing.So the model actually understood it fine, the checking method was just too strict.

2. Reliability: I ran the extractor 5 times at temperature 0 and 5 times at temperature 1 on the same letter. All 10 runs gave the exact same answer. This is a good sign, but I would still use temperature=0 in a real system, because it removes randomness completely instead of hoping the letter stays easy to read.

3. Hallucination: No, the system did not make things up. When I asked about something not in the letter (credit score), it correctly said it wasn't there. When I gave it an unrelated weather report, it correctly returned null for everything instead of inventing a fake applicant. To make this even safer in real use, I would add a check that confirms every extracted value actually appears in the original letter text.

## Part 4.4

1. Who could be unfairly harmed by full automation:
Some real, solid business owners write in simple or broken English, use short sentences, or don't explain their numbers well, not because their business is weak, but because they didn't get much formal education. If the system judges people partly by how well-written their letter is, these applicants could be marked as "risky" even though their business is actually doing fine. This would unfairly hurt honest traders just because of how they write, not how they work.

2. Sending personal data to a third-party API in another country:
* Loan letters contain private information such as names, income, business details. When we send this to an API like Groq, the data leaves Ghana and goes to servers in another country, which may have different privacy laws. Before using this in a real microfinance institution, I would check: Does the API company store or use our data to train their models? Does this follow Ghana's Data Protection Act? Have the applicants agreed to their information being sent outside the country? These are important questions to answer before trusting a real system with real people's data.

3. Two safeguards I would add in production:

* Human review before any decision: A real loan officer must always read the full letter and confirm the AI's brief before anything is decided. The AI should only help, never decide alone.
Logging and an appeal process: Every letter, prompt, and AI output should be saved and reviewed regularly for mistakes or unfair patterns. If an applicant feels the system misunderstood their letter, they should be able to ask a human to look at it again.

## Reflection Questions

#####1
Changing a prompt and changing a hyperparameter are similar in one way, both are "try it, see what happens, then adjust" loops. But they're very different in speed and cost. If I change a hyperparameter (like learning rate) in Lab 3, I have to retrain the whole neural network again, which takes real time and computing power. If I change a prompt, I just rewrite the words and run it again, no training needed, the answer comes back in seconds. So prompting is a much faster, cheaper way to "tune" a system.

#### 2

After doing Section 4, I would trust this system to help a loan officer, but not to run completely alone. The result that influenced me most was the hallucination test because both times I tried to trick it (asking about a missing credit score, and giving it a weather report instead of a letter), it did not make anything up. It honestly said "not stated" and returned null instead of inventing an answer. That gave me the most confidence. But I still wouldn't let it run unattended, because it can't judge things a human can, like whether someone is being honest, or whether a business idea is realistic.

####3

From my Part 1.1 test call, one small question used 375 tokens total. In the real system, each loan application isn't just one call, it needs three: a summary, an extraction, and a brief. If each of those calls uses roughly 300–500 tokens, one full application could use around 1,000–1,500 tokens.

For 1,000 applications a month, that's about 1,000,000 to 1,500,000 tokens a month. That's a lot, but Groq's free tier is generous and fast, so it could likely still handle a project at this scale, or close to it. If a real bank wanted to process many more applications, or wanted more reliability guarantees, they'd need to check Groq's paid pricing or compare it to other providers, since free tiers usually have limits meant for testing and small projects, not full business use.

####4.  Why an API beats training your own model

Using an API is faster and easier for this task, because the model already knows how to read and understand language well. I didn't have to collect data or train anything myself. I just wrote good prompts. This is very different from Lab 3, where I had to build and train a neural network from scratch just to classify one type of data.

But an API isn't always the better choice. Training your own model could be better if: the data is too private to send to an outside company, the system needs to work without internet access, or the task is very specific and needs to run millions of times cheaply, in that case, a smaller custom-trained model could end up cheaper and faster than paying per API call.